# TinyCeNN Qwen3.5 — Optimized Standalone Release

This notebook creates **standalone, validated Hugging Face releases** for:

- `vtava/Qwen3.5-0.8B-MemoryFusion`
- `vtava/Qwen3.5-0.8B-CeNN-Integrated-V1`
- `vtava/Qwen3.5-0.8B-PDelta3-CLVR-Local32`

It intentionally does **not** build llama.cpp or GGUF. Each model is reconstructed with its TinyCeNN layers, runs a small QuickCheck, is saved as complete `model.safetensors`, then reloaded in a fresh Python process using only the minimal runtime bundled into the release. Upload happens only after that reload passes.

> Use a GPU runtime. Add a Colab secret named `HF_TOKEN` with **write** permission.


In [ ]:
#@title 1. Install minimal runtime + update TinyCeNN-LM
%pip -q install -U huggingface_hub safetensors accelerate sentencepiece
%pip -q install -U "transformers @ git+https://github.com/huggingface/transformers.git@main"

from pathlib import Path
import subprocess, sys

ROOT = Path('/content')
TINY = ROOT / 'TinyCeNN-LM'
WORK = ROOT / 'tinycenn_standalone'

if TINY.exists():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=TINY, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(TINY)], check=True)

# Verify Qwen3.5 support in a fresh interpreter (avoids stale notebook imports).
check = subprocess.run([
    sys.executable, '-c',
    'import transformers; import transformers.models.qwen3_5; '
    'from transformers import Qwen3_5ForCausalLM; '
    'print("Transformers", transformers.__version__, "| Qwen3.5 OK")'
], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(check.stdout)
if check.returncode:
    raise RuntimeError('Qwen3.5 Transformers verification failed')

print('TinyCeNN commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=TINY, text=True).strip())
print('GPU available:', __import__('torch').cuda.is_available())


In [ ]:
#@title 2. Hugging Face login
import os
from getpass import getpass
from huggingface_hub import HfApi, login

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not token:
    token = getpass('HF write token: ').strip()
if not token:
    raise RuntimeError('HF_TOKEN is required')

login(token=token, add_to_git_credential=False)
os.environ['HF_TOKEN'] = token
print('Authenticated as:', HfApi(token=token).whoami()['name'])


In [ ]:
#@title 3. Build, QuickCheck, standalone reload-test, and upload all 3
import os, subprocess, sys

cmd = [
    sys.executable, '-u', str(TINY / 'scripts/qwen35_standalone_release.py'),
    '--work-dir', str(WORK),
    '--only', 'all',
    '--upload',
]

env = os.environ.copy()
env['HF_TOKEN'] = token
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd))
process = subprocess.Popen(
    cmd, cwd=TINY, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
rc = process.wait()
print('\nRelease process return code:', rc)
if not (WORK / 'summary.json').exists():
    raise RuntimeError('Release script did not create summary.json')


In [ ]:
#@title 4. Final report
import json
import pandas as pd
from IPython.display import display

results = json.loads((WORK / 'summary.json').read_text())
rows = []
for r in results:
    rows.append({
        'Source': r.get('source'),
        'Variant': r.get('variant'),
        'Reload': r.get('standalone_reload'),
        'QuickCheck %': (r.get('quickcheck') or {}).get('score'),
        'Custom layers': r.get('custom_layers'),
        'Custom tensors': r.get('custom_tensor_count'),
        'Uploaded': r.get('uploaded'),
        'URL / Error': r.get('url') or r.get('error'),
    })
display(pd.DataFrame(rows))

passed = [r for r in results if r.get('standalone_reload') == 'PASS']
failed = [r for r in results if r.get('standalone_reload') != 'PASS']
print(f'\nStandalone PASS: {len(passed)}/{len(results)}')
for r in passed:
    print('✅', r.get('url') or r.get('target'))
for r in failed:
    print('❌', r.get('source'), '->', r.get('error'))
print('\nFull report:', WORK / 'summary.json')
